# Spiking Transformer U-Net for BraTS 2023

This notebook implements a complete standalone training pipeline for the BraTS 2023 Glioma dataset using a custom Spiking Transformer U-Net architecture. It incorporates all 6 phases requested:
1. **3D NIfTI Data Ingestion & Target Engineering**
2. **Spiking Tokenizer**
3. **Agentic Triage (Adaptive Inference)**
4. **Spiking Transformer Encoder (High-Precision Mode with TSN, Shiftmax, BSA)**
5. **Spiking U-Net Decoder (Event-Driven Skip Connections)**
6. **FPTT Optimization (Memory Wall Bypass)**


In [ ]:
## mount drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q snntorch nibabel

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nibabel as nib
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Phase 1: 3D NIfTI Data Ingestion & Target Engineering

In [ ]:
# Constants
DATA_DIR = "/content/drive/MyDrive/datasets/brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
PATCH_SIZE = (96, 96, 96)

class BraTS3DDataset(Dataset):
    def __init__(self, patient_dirs, patch_size=PATCH_SIZE):
        self.patient_dirs = patient_dirs
        self.patch_size = patch_size

    def __len__(self):
        return len(self.patient_dirs)

    def __getitem__(self, idx):
        patient_dir = self.patient_dirs[idx]
        
        # Find NIfTI files by sequence
        t1n_path = glob.glob(os.path.join(patient_dir, "*-t1n.nii*"))[0]
        t1c_path = glob.glob(os.path.join(patient_dir, "*-t1c.nii*"))[0]
        t2w_path = glob.glob(os.path.join(patient_dir, "*-t2w.nii*"))[0]
        t2f_path = glob.glob(os.path.join(patient_dir, "*-t2f.nii*"))[0]
        seg_path = glob.glob(os.path.join(patient_dir, "*-seg.nii*"))[0]
        
        # Load Modalities
        t1n = nib.load(t1n_path).get_fdata().astype(np.float32)
        t1c = nib.load(t1c_path).get_fdata().astype(np.float32)
        t2w = nib.load(t2w_path).get_fdata().astype(np.float32)
        t2f = nib.load(t2f_path).get_fdata().astype(np.float32)
        seg = nib.load(seg_path).get_fdata().astype(np.float32)
        
        # Stack into 4-channels: (C, D, H, W)
        image_4d = np.stack([t1n, t1c, t2w, t2f], axis=0)
        image_4d = np.transpose(image_4d, (0, 3, 1, 2))
        seg_3d = np.transpose(seg, (2, 0, 1))
                
        d, h, w = image_4d.shape[1:]
        pd, ph, pw = self.patch_size
        
        if d < pd or h < ph or w < pw:
            image_4d = np.pad(image_4d, ((0,0), (0, max(0, pd-d)), (0, max(0, ph-h)), (0, max(0, pw-w))))
            seg_3d = np.pad(seg_3d, ((0, max(0, pd-d)), (0, max(0, ph-h)), (0, max(0, pw-w))))
            d, h, w = image_4d.shape[1:]

        # 3D Patching Constraints: Random Crop
        z = np.random.randint(0, d - pd + 1)
        y = np.random.randint(0, h - ph + 1)
        x = np.random.randint(0, w - pw + 1)
        
        image_patch = image_4d[:, z:z+pd, y:y+ph, x:x+pw]
        seg_patch = seg_3d[z:z+pd, y:y+ph, x:x+pw]
        
        # Label Mapping
        wt_mask = (seg_patch > 0).astype(np.float32)
        tc_mask = np.logical_or(seg_patch == 1, seg_patch == 3).astype(np.float32)
        et_mask = (seg_patch == 3).astype(np.float32)
        
        mask_channels = np.stack([wt_mask, tc_mask, et_mask], axis=0) 
        
        # Z-score normalization per channel
        for c in range(4):
            img_c = image_patch[c]
            if img_c.max() > 0:
                mean, std = img_c[img_c > 0].mean(), img_c[img_c > 0].std()
                img_c = (img_c - mean) / (std + 1e-8)
                image_patch[c] = img_c

        return {
            'image': torch.from_numpy(image_patch),
            'mask': torch.from_numpy(mask_channels)
        }


## Phase 2 & 4: Core Spiking Building Blocks

In [ ]:
class TernarySurrogate(torch.autograd.Function):
    """
    Custom Surrogate Gradient for Ternary Spikes (-1, 0, 1).
    """
    @staticmethod
    def forward(ctx, input, threshold=1.0):
        ctx.save_for_backward(input)
        ctx.threshold = threshold
        out = torch.zeros_like(input)
        out[input >= threshold] = 1.0
        out[input <= -threshold] = -1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        threshold = ctx.threshold
        # Exponential surrogate taking absolute magnitude
        surrogate_grad = torch.exp(-torch.abs(input))
        return grad_output * surrogate_grad, None

def ternary_spike(x, threshold=1.0):
    return TernarySurrogate.apply(x, threshold)

class TernaryLIF(nn.Module):
    """
    Ternary Spiking Neurons (TSN) cell, tracking multi-polar membrane potential.
    """
    def __init__(self, beta=0.9, threshold=1.0):
        super().__init__()
        self.beta = beta
        self.threshold = threshold
        self.register_buffer("mem", None)
        
    def reset_state(self):
        self.mem = None

    def detach_state(self):
        if self.mem is not None:
            self.mem.detach_()

    def forward(self, x):
        if self.mem is None or self.mem.shape != x.shape:
            self.mem = torch.zeros_like(x)
            
        self.mem = self.beta * self.mem + x
        spk = ternary_spike(self.mem, self.threshold)
        
        # Soft reset by subtraction
        self.mem = self.mem - spk * self.threshold
        return spk


In [ ]:
def shiftmax(x, dim=-1):
    """
    Shiftmax Normalization replacing standard Softmax.
    Approximates exp(x) / sum(exp(x)) purely using shifts (powers of 2).
    """
    x_max = torch.max(x, dim=dim, keepdim=True)[0]
    x_norm = x - x_max
    
    # We substitute exp(x) with 2^(floor(x_norm)) which is analogous to a bit-shift 
    approx_exp = 2.0 ** torch.floor(x_norm)
    denom = torch.sum(approx_exp, dim=dim, keepdim=True) + 1e-8
    
    return approx_exp / denom

class BipolarSelfAttention(nn.Module):
    """
    Bipolar Self-Attention (BSA) with shiftmax.
    """
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        
        self.lif_q = TernaryLIF()
        self.lif_k = TernaryLIF()
        self.lif_v = TernaryLIF()
        
    def reset_states(self):
        self.lif_q.reset_state()
        self.lif_k.reset_state()
        self.lif_v.reset_state()

    def forward(self, x):
        # Processing purely spatial vectors per time step t
        B, S, E = x.shape
        
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        
        q_spk = self.lif_q(q)
        k_spk = self.lif_k(k)
        v_spk = self.lif_v(v)
        
        q_head = q_spk.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k_head = k_spk.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v_head = v_spk.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
        attn_scores = torch.matmul(q_head, k_head.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = shiftmax(attn_scores, dim=-1)
        
        attn_out = torch.matmul(attn_weights, v_head)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, E)
        
        return self.out_proj(attn_out)


## Phase 2 & 5: Spiking Tokenizer and U-Net Components

In [ ]:
class SpikingTokenizer3D(nn.Module):
    def __init__(self, in_channels=4, embed_dim=32):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, embed_dim // 2, kernel_size=3, stride=2, padding=1)
        self.lif1 = TernaryLIF()
        self.conv2 = nn.Conv3d(embed_dim // 2, embed_dim, kernel_size=3, stride=2, padding=1)
        self.lif2 = TernaryLIF()
        
    def reset_states(self):
        self.lif1.reset_state()
        self.lif2.reset_state()

    def forward(self, x):
        c1 = self.conv1(x)
        s1 = self.lif1(c1)
        c2 = self.conv2(s1)
        s2 = self.lif2(c2)
        return s2, s1

class SpikingTransformerEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, depth):
        super().__init__()
        self.layers = nn.ModuleList([
            BipolarSelfAttention(embed_dim, num_heads) for _ in range(depth)
        ])
        
    def reset_states(self):
        for layer in self.layers:
            layer.reset_states()

    def forward(self, x):
        B, E, D, H, W = x.shape
        # Flatten spatial dimensions for Transformer attention
        x_flat = x.view(B, E, D*H*W).transpose(1, 2) 
        for layer in self.layers:
            x_flat = x_flat + layer(x_flat)
        return x_flat.transpose(1, 2).view(B, E, D, H, W)

class SpikingUNetDecoder3D(nn.Module):
    def __init__(self, embed_dim, out_channels=3):
        super().__init__()
        self.upconv1 = nn.ConvTranspose3d(embed_dim, embed_dim // 2, kernel_size=4, stride=2, padding=1)
        self.lif_up1 = TernaryLIF()
        self.upconv2 = nn.ConvTranspose3d(embed_dim // 2, out_channels, kernel_size=4, stride=2, padding=1)
        # Last layer evaluates membrane potentials continuously for logits/loss, no spikes emitted
        
    def reset_states(self):
        self.lif_up1.reset_state()

    def forward(self, encoder_spk, skip_spk):
        u1 = self.upconv1(encoder_spk)
        
        # Event-Driven Skip Connection (fuse upsampled tokens with pristine token spikes)
        # Align spatial dimensions (both are [B, E/2, D/2, H/2, W/2])
        u1_fused = u1 + skip_spk
        s_u1 = self.lif_up1(u1_fused)
        
        u2 = self.upconv2(s_u1)
        return u2


## Phase 3: Agentic Triage (Adaptive Inference)

In [ ]:
class AgenticSpikingTransformer(nn.Module):
    def __init__(self, in_channels=4, out_channels=3, embed_dim=32, num_heads=4, depth=2, entropy_threshold=0.5):
        super().__init__()
        self.tokenizer = SpikingTokenizer3D(in_channels, embed_dim)
        self.encoder = SpikingTransformerEncoder(embed_dim, num_heads, depth)
        self.decoder = SpikingUNetDecoder3D(embed_dim, out_channels)
        self.entropy_threshold = entropy_threshold

    def reset_all_states(self):
        self.tokenizer.reset_states()
        self.encoder.reset_states()
        self.decoder.reset_states()
        
    def forward_step(self, x):
        enc_spk, skip_spk = self.tokenizer(x)
        trans_spk = self.encoder(enc_spk)
        out_mem = self.decoder(trans_spk, skip_spk)
        return out_mem

    def forward(self, x, force_T=None):
        # Force T ignores triage (used purely for validation/FPTT unroll)
        if force_T is not None:
            out_potentials = []
            self.reset_all_states()
            for t in range(force_T):
                out_potentials.append(self.forward_step(x))
            return torch.stack(out_potentials, dim=0)
        
        # Agentic Triage Logic for Dynamic High-Precision Inference
        self.reset_all_states()
        
        # Low-Power Screening Mode (T=1)
        mem_t1 = self.forward_step(x)
        probs_t1 = torch.sigmoid(mem_t1)
        
        # Calculate Predictive Entropy Score via Binary Cross Entropy property
        entropy = -probs_t1 * torch.log(probs_t1 + 1e-6) - (1-probs_t1) * torch.log(1-probs_t1 + 1e-6)
        mean_entropy = entropy.mean().item()
        
        if mean_entropy > self.entropy_threshold:
            # Complex boundaries -> Dynamic Routing to High-Precision Mode (T=4)
            mem_sum = mem_t1
            for _ in range(3):
                mem_sum += self.forward_step(x)
            return (mem_sum / 4.0).unsqueeze(0)
        else:
            # Simple structures -> Fast Exit, retains Low-Power configuration (T=1)
            return mem_t1.unsqueeze(0)


## Phase 6: FPTT Optimization Loop & Training Execution

In [ ]:
def detach_states(model):
    """
    Memory Wall Bypass for 3D SNNs via Forward Propagation Through Time.
    Explicitly disconnects membrane hidden states from the computational graph to release memory.
    """
    for module in model.modules():
        if isinstance(module, TernaryLIF):
            module.detach_state()

def train_fptt(model, dataloader, optimizer, criterion, epochs=5, T_train=4, trunc_steps=2):
    model.to(device)
    model.train()
    
    for epoch in range(epochs):
        epoch_loss = 0
        
        for batch_idx, batch in enumerate(dataloader):
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            
            model.reset_all_states()
            batch_loss = 0
            loss_accum = 0
            optimizer.zero_grad()
            
            for t in range(T_train):
                # Step-wise event-driven evaluation
                out_mem = model.forward_step(images)
                loss = criterion(out_mem, masks)
                loss_accum += loss
                batch_loss += loss.item()
                
                # FPTT logic: Truncate every `trunc_steps`
                if (t + 1) % trunc_steps == 0 or (t + 1) == T_train:
                    loss_accum.backward() 
                    optimizer.step()
                    optimizer.zero_grad()
                    loss_accum = 0
                    
                    detach_states(model)
                    
            epoch_loss += batch_loss / T_train
            print(f"Epoch [{epoch+1}/{epochs}] | Batch {batch_idx+1}/{len(dataloader)} | Avg Loss: {batch_loss/T_train:.4f}")
            
        print(f"==> Epoch {epoch+1} Completed. Loss: {epoch_loss/len(dataloader):.4f}")


In [ ]:
# Execution Block (Settings mapping to Colab architecture)
patient_dirs = [os.path.join(DATA_DIR, d) for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]

if len(patient_dirs) > 0:
    # Small subset representing the loader, adjust logic to partition out Validation sets directly
    train_dataset = BraTS3DDataset(patient_dirs=patient_dirs[:4]) 
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

    model = AgenticSpikingTransformer(in_channels=4, out_channels=3, embed_dim=32, num_heads=4, depth=2)
    
    # Binary Cross Entropy over the 3 sub-channels (WT, TC, ET)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    print("Starting FPTT Memory-efficient Training Loop for 3D Volume Segmentation...")
    train_fptt(model, train_loader, optimizer, criterion, epochs=2, T_train=4, trunc_steps=2)
else:
    print("Dataset path not valid or empty. Please mount the google drive securely and retry.")
print("。.:☆*:・'(*⌒―⌒*))) Training Pipeline is successfully configured!")
